<a href="https://colab.research.google.com/github/mohsen-alshoklia/Manim-CE-files/blob/main/mathematical_functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!sudo apt update
!sudo apt install libcairo2-dev ffmpeg texlive texlive-latex-extra texlive-fonts-extra texlive-latex-recommended texlive-science tipa libpango1.0-dev
!pip install manim

In [29]:
from manim import *
import numpy as np
from IPython.display import Video, display
import subprocess
import os

class FunctionShowcase(Scene):
    def construct(self):
        # Define all the functions to showcase
        functions_data = [
            {'type': 'explicit', 'func': lambda x: x, 'domain': [-2.5, 2.5], 'tex': r"y = x", 'color': PINK},
            {'type': 'explicit', 'func': lambda x: x**2, 'domain': [-2.2, 2.2], 'tex': r"y = x^2", 'color': TEAL},
            {'type': 'explicit', 'func': lambda x: x**3, 'domain': [-1.7, 1.7], 'tex': r"y = x^3", 'color': GOLD},
            {'type': 'explicit', 'func': lambda x: x**5, 'domain': [-1.35, 1.35], 'tex': r"y = x^5", 'color': BLUE},
            {'type': 'explicit', 'func': lambda x: np.sqrt(x), 'domain': [0, 4], 'tex': r"y = \sqrt{x}", 'color': ORANGE},
            {'type': 'explicit', 'func': lambda x: 2*np.sin(2*x), 'domain': [-3, 3], 'tex': r"y = 2\sin(2x)", 'color': BLUE},
            {'type': 'explicit', 'func': lambda x: 2*np.cos(2*x), 'domain': [-3, 3], 'tex': r"y = 2\cos(2x)", 'color': GREEN},
            {'type': 'explicit', 'func': lambda x: np.exp(-x), 'domain': [-1.6, 4], 'tex': r"y = e^{-x}", 'color': RED},
            {'type': 'explicit', 'func': lambda x: (2.6*x)/(1 + x**2), 'domain': [-4, 4], 'tex': r"y = \frac{2.6x}{1 + x^2}", 'color': PURPLE},
            # New functions added here
            {'type': 'explicit', 'func': lambda x: x * np.sin(1/x) if x != 0 else 0, 'domain': [-0.5, 0.5], 'tex': r"y = x\sin\left(\frac{1}{x}\right)", 'color': MAROON},
            {'type': 'explicit', 'func': lambda x: 8/(x**2 + 4), 'domain': [-5, 5], 'tex': r"y = \frac{8}{x^2 + 4}", 'color': TEAL},
            {'type': 'explicit', 'func': lambda x: np.exp(-abs(x)) * np.sin(10*x), 'domain': [-2, 2], 'tex': r"y = e^{-|x|}\sin(10x)", 'color': LIGHT_BROWN},
            {'type': 'explicit', 'func': lambda x: np.sin(x) + 0.5*np.sin(3*x) + 0.25*np.sin(5*x), 'domain': [-6, 6], 'tex': r"y = \sin(x) + \frac{1}{2}\sin(3x) + \frac{1}{4}\sin(5x)", 'color': DARK_BLUE},
            {'type': 'explicit', 'func': lambda x: np.sin(x**2), 'domain': [-5, 5], 'tex': r"y = \sin(x^2)", 'color': GREEN_E},
            {'type': 'parametric', 'func': lambda t: np.array([2*np.sin(4*t)*np.cos(t), 2*np.sin(4*t)*np.sin(t), 0]), 'domain': [0, 2*np.pi], 'tex': r"r = 2\sin(4\theta)", 'color': GREEN},
            {'type': 'implicit', 'func': lambda x, y: x**6 + y**6 - 2, 'domain': [-1.5, 1.5], 'tex': r"x^6 + y^6 = 2", 'color': ORANGE},
            {'type': 'implicit', 'func': lambda x, y: (x**2 + y**2)**2 - 2.4*(x**2 - y**2), 'domain': [-1.8, 1.8], 'tex': r"(x^2+y^2)^2 = 2.4(x^2-y^2)", 'color': YELLOW},
            # Heart function
            {'type': 'implicit', 'func': lambda x, y: (x**2 + y**2 - 1)**3 - x**2 * y**3, 'domain': [-1.5, 1.5], 'tex': r"(x^2 + y^2 - 1)^3 = x^2 y^3", 'color': RED},
        ]

        # Create title with gradient
        title = Text("Function Showcase", font_size=48, weight=BOLD).move_to(5.5*UP)
        gradient_colors = [BLUE, PURPLE, PINK, RED, ORANGE, YELLOW]
        title.set_color_by_gradient(gradient_colors)

        # Display title
        self.play(Write(title, lag_ratio=0.0), run_time=1)
        self.wait(0.3)
        # self.play(FadeOut(title), run_time=0.5)

        # Create axes for explicit functions
        axes_explicit = Axes(
            x_range=[-4, 4, 1],
            y_range=[-3, 3, 1],
            x_length=7,
            y_length=6,
            tips=True,
            axis_config={"color": WHITE, "include_ticks": True, "tick_size": 0.05, "stroke_width": 1}
        )
        axes_explicit.move_to(ORIGIN)

        axis_labels_explicit = axes_explicit.get_axis_labels(x_label="x", y_label="y")
        axis_labels_explicit[0].scale(0.5)
        axis_labels_explicit[1].scale(0.5)

        # Create axes for implicit/parametric functions
        axes_implicit = Axes(
            x_range=[-2.5, 2.5, 1],
            y_range=[-2.5, 2.5, 1],
            x_length=7,
            y_length=6,
            tips=True,
            axis_config={"color": WHITE, "include_ticks": True, "tick_size": 0.05, "stroke_width": 1}
        )
        axes_implicit.move_to(ORIGIN)

        axis_labels_implicit = axes_implicit.get_axis_labels(x_label="x", y_label="y")
        axis_labels_implicit[0].scale(0.5)
        axis_labels_implicit[1].scale(0.5)

        current_axes = None
        current_axes_obj = None
        current_labels = None

        # Helper function to create equation with box
        def create_equation_with_box(tex, color):
            equation = MathTex(tex, font_size=32, color=color)
            box = RoundedRectangle(
                width=equation.width + 0.8,
                height=equation.height + 0.5,
                corner_radius=0.2,
                color=color,
                stroke_width=3,
                fill_opacity=0.15,
                fill_color=color
            )
            # Position box to surround the equation
            box.move_to(equation.get_center())
            group = VGroup(box, equation)
            return group

        # Create initial equation box
        current_equation_group = create_equation_with_box(functions_data[0]['tex'], functions_data[0]['color'])
        current_equation_group.move_to(3.5*DOWN)

        # Store initial graph and equation
        current_graph = None

        # Process each function
        for i, func_data in enumerate(functions_data):
            func_type = func_data['type']
            func_color = func_data['color']
            func_tex = func_data['tex']

            # Choose appropriate axes
            if func_type in ['implicit', 'parametric']:
                axes = axes_implicit
                axis_labels = axis_labels_implicit
                if current_axes != 'implicit':
                    if current_axes_obj:
                        self.play(
                            Transform(current_axes_obj, axes),
                            Transform(current_labels, axis_labels),
                            run_time=0.5
                        )
                    else:
                        self.play(Create(axes), Write(axis_labels), run_time=0.5)
                        current_axes_obj = axes
                        current_labels = axis_labels
                    current_axes = 'implicit'
            else:
                axes = axes_explicit
                axis_labels = axis_labels_explicit
                if current_axes != 'explicit':
                    if current_axes_obj:
                        self.play(
                            Transform(current_axes_obj, axes),
                            Transform(current_labels, axis_labels),
                            run_time=0.5
                        )
                    else:
                        self.play(Create(axes), Write(axis_labels), run_time=0.5)
                        current_axes_obj = axes
                        current_labels = axis_labels
                    current_axes = 'explicit'

            # Create new equation with box
            new_equation_group = create_equation_with_box(func_tex, func_color)
            new_equation_group.move_to(3.5*DOWN)

            # Create new graph
            new_graph = None

            if func_type == 'explicit':
                domain = func_data.get('domain', [-3, 3])
                new_graph = axes.plot(
                    func_data['func'],
                    color=func_color,
                    stroke_width=3,
                    x_range=domain
                )

            elif func_type == 'implicit':
                new_graph = ImplicitFunction(
                    lambda x, y: func_data['func'](x, y),
                    x_range=axes.x_range[:2],
                    y_range=axes.y_range[:2],
                    color=func_color,
                    stroke_width=3
                )

            elif func_type == 'parametric':
                domain = func_data.get('domain', [0, 2*np.pi])
                new_graph = axes.plot_parametric_curve(
                    func_data['func'],
                    t_range=domain,
                    color=func_color,
                    stroke_width=3
                )

            # Animate transformations
            if i == 0:
                # First function: create everything
                self.play(Create(axes), Write(axis_labels), run_time=0.5)
                self.play(Create(new_graph), run_time=0.8)
                self.play(Write(current_equation_group), run_time=0.5)
                current_graph = new_graph
            else:
                # Transform graph and equation simultaneously
                self.play(
                    Transform(current_graph, new_graph),
                    Transform(current_equation_group, new_equation_group),
                    run_time=0.2
                )

            self.wait(0.5)  # Brief pause to appreciate each function
        self.wait(0.4)

# Render with high quality, vertical format
if __name__ == "__main__":
    from manim import config as manim_config

    # Configure for YouTube Shorts / WhatsApp Status (9:16 vertical format)
    manim_config.frame_width = 9
    manim_config.frame_height = 16
    manim_config.pixel_height = 1920
    manim_config.pixel_width = 1080


    scene = FunctionShowcase()
    scene.render()

    video_path = scene.renderer.file_writer.movie_file_path

    # Display video info
    try:
        result = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0',
                                '-show_entries', 'stream=width,height,r_frame_rate', '-of', 'csv=p=0',
                                video_path], capture_output=True, text=True)
        dimensions = result.stdout.strip()
        print(f"Video info: {dimensions}")
        print(f"Expected: 1080x1920 at 60 FPS")
    except:
        pass

    # Display the video
    display(Video(video_path, embed=True))

Video info: 1080,1920,60/1
Expected: 1080x1920 at 60 FPS
